# Procesamiento de Indicadores de Interculturalidad

Este notebook consolida la información de autoidentificación cultural de Estudiantes, Docentes y Padres/Cuidadores para su análisis comparativo en Power BI.

**Indicador:** % de participantes que se autoidenitifican con una nación o pueblo indígena originario.

In [1]:
import pandas as pd
import os

# Configuración de rutas
RAW_DOCENTES = '../raw_data/BBDD_Docentes_dash.xlsx'
RAW_PADRES = '../raw_data/BBDD_Padres_dash.xlsx'
OUTPUT_ESTUDIANTES = '../output/df_estudiantes_indicadores_COMPLETO.xlsx'
FINAL_OUTPUT = '../output/df_consolidado_interculturalidad.xlsx'

print("Cargando datos...")
df_doc = pd.read_excel(RAW_DOCENTES)
df_pad = pd.read_excel(RAW_PADRES)
df_est = pd.read_excel(OUTPUT_ESTUDIANTES)
print("Datos cargados correctamente.")

Cargando datos...


Datos cargados correctamente.


## 1. Mapeo y Homologación de Columnas

In [2]:
# 1.1 Procesamiento DOCENTES
doc_cols = {
    'Municipio': 'Municipio',
    'Unidad educativa': 'Unidad_Educativa',
    'D4. ¿Cuál es su género?': 'Genero',
    'D5 ¿Se identifica como parte de alguna nación o pueblo indígena originario campesino?': 'Autoidentificacion',
    'Si se identifica como parte de alguna nación o pueblo indígena originario campesino indique cual': 'Nacion_Pueblo'
}
df_doc_sub = df_doc[list(doc_cols.keys())].rename(columns=doc_cols)
df_doc_sub['Grupo'] = 'Docentes'

# 1.2 Procesamiento PADRES
pad_cols = {
    'Municipio': 'Municipio',
    'Unidad Educativa': 'Unidad_Educativa',
    'P2. ¿Cual es su Género?': 'Genero',
    'P7. ¿Se identifica como parte de alguna nación o pueblo indígena originario campesino?': 'Autoidentificacion',
    'Si se identifica como parte de alguna nación o pueblo indígena originario campesino indique cual:': 'Nacion_Pueblo'
}
df_pad_sub = df_pad[list(pad_cols.keys())].rename(columns=pad_cols)
df_pad_sub['Grupo'] = 'Padres'

# 1.3 Procesamiento ESTUDIANTES
est_cols = {
    'Municipio': 'Municipio',
    'Unidad Educativa': 'Unidad_Educativa',
    'genero_clean': 'Genero',
    'Autoidentificación cultural': 'Autoidentificacion',
    'Si respondiste "Si" a la auto identificación cultural, indica cual': 'Nacion_Pueblo'
}
df_est_sub = df_est[list(est_cols.keys())].rename(columns=est_cols)
df_est_sub['Grupo'] = 'Estudiantes'

# Consolidación inicial
df_inter = pd.concat([df_doc_sub, df_pad_sub, df_est_sub], ignore_index=True)
df_inter.head()

,Municipio,Unidad_Educativa,Genero,Autoidentificacion,Nacion_Pueblo,Grupo
0,La Paz,Unidad Educativa Carlos Medinacelli,Femenino,Sí,Quechua,Docentes
1,La Paz,Unidad Educativa República de Japón,Masculino,No,NaN,Docentes
2,La Paz,Unidad Educativa Carlos Medinacelli,Femenino,Sí,Aymara,Docentes
3,La Paz,Unidad Educativa Carlos Medinacelli,Femenino,No,Ninguna,Docentes
4,La Paz,Unidad Educativa República de Japón,Femenino,Sí,Aymara,Docentes


## 2. Limpieza y Enriquecimiento de Datos

In [3]:
def clean_si_no(val):
    if pd.isna(val): return "No responde"
    val = str(val).strip().lower()
    if val in ['sí', 'si', 'sì', 'si ']: return 'Sí'
    if val in ['no', 'nó', 'no ']: return 'No'
    return "No responde"

df_inter['Autoidentificacion'] = df_inter['Autoidentificacion'].apply(clean_si_no)

# 2.1 Indicador Numérico
df_inter['Autoidentificacion_Numeric'] = df_inter['Autoidentificacion'].map({'Sí': 1, 'No': 0, 'No responde': 0})

# 2.2 Georreferenciación (Texto y Coordenadas)
df_inter['Ubicacion_Mapa'] = df_inter['Municipio'].fillna('') + ", Bolivia"

coords = {
    'La Paz': {'Latitud': -16.5000, 'Longitud': -68.1333},
    'El Alto': {'Latitud': -16.5167, 'Longitud': -68.2000},
    'Palca': {'Latitud': -16.5611, 'Longitud': -67.9531},
    'Villa Montes': {'Latitud': -21.2608, 'Longitud': -63.4761},
    'Santa Cruz': {'Latitud': -17.7861, 'Longitud': -63.1811}
}

df_inter['Latitud'] = df_inter['Municipio'].map(lambda x: coords.get(x, {}).get('Latitud'))
df_inter['Longitud'] = df_inter['Municipio'].map(lambda x: coords.get(x, {}).get('Longitud'))

def clean_genero(val):
    if pd.isna(val): return "No responde"
    val = str(val).strip().lower()
    if 'mujer' in val or 'femenino' in val: return 'Mujer'
    if 'hombre' in val or 'varón' in val or 'varon' in val or 'masculino' in val: return 'Hombre'
    return "Otro/No responde"

df_inter['Genero'] = df_inter['Genero'].apply(clean_genero)

# Limpieza de Nacion_Pueblo
df_inter['Nacion_Pueblo'] = df_inter['Nacion_Pueblo'].fillna('Ninguno').str.strip().str.title()

print("Procesamiento completado con Ubicación de Mapa y Coordenadas.")

Procesamiento completado con Ubicación de Mapa y Coordenadas.


## 3. Exportación para Power BI

In [4]:
df_inter.to_excel(FINAL_OUTPUT, index=False)
print(f"Archivo guardado en: {FINAL_OUTPUT}")

Archivo guardado en: ../output/df_consolidado_interculturalidad.xlsx
